# Run inference for Nemotron H + EarTTS

In [ ]:
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

import os
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"


# audio tokens for prompt to derive speaker identity
prompt_audio_codes = torch.load("eartts_debug_tokens/code.pt").cpu().to(torch.int32)
prompt_audio_codes = torch.nn.functional.pad(prompt_audio_codes[0, :-1, :], (0, 0, 1, 0))  # T x 31
# subword ids corresponding to the text to synthesize
next_subword_ids = torch.load("eartts_debug_tokens/next_subword_ids.pt").cpu().to(torch.int32)[0]  # T
# subword ids corresponding to the prompt, instruction, etc.
subword_ids_prompt = torch.load("eartts_debug_tokens/input_text_tokens.pt").cpu().to(torch.int32)[0]  # T
last_prompt_subword_id = subword_ids_prompt[-1:]  # 1
subword_ids_prompt = subword_ids_prompt[:-1]  # T


# load vllm engine
type_str = "bfloat16"
torch_type = getattr(torch, type_str)
engine_args = AsyncEngineArgs(
    model="eartts_vllm_model",
    dtype=type_str,
    max_model_len=256,
    max_num_batched_tokens=512,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using embeddings directly
    load_format="dummy",
    #enforce_eager=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=64, skip_sampling=True)


bos_mask = torch.zeros_like(subword_ids_prompt, dtype=torch_type)
input_acoustic_embeds = torch.randn(subword_ids_prompt.shape[0], 4480, dtype=torch_type)
bos_mask[0] = 1.0
inputs = {
    # dummy tokens
    "prompt_token_ids": [0] * prompt_audio_codes.shape[0],
    # actual inpust to the model in prefill stage
    "custom_inputs": {
        "input_acoustic_embeds": input_acoustic_embeds,
        "acoustic_tokens": prompt_audio_codes,
        "context_text_tokens": subword_ids_prompt,
        # only context is used in prefill stage, mask is not used
        "text_mask": torch.zeros_like(subword_ids_prompt, dtype=torch_type),
        "bos_mask": bos_mask,
    }
}


acoustic_tokens_lst = []
i = 0
context_subword_id = last_prompt_subword_id  # (1,)
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id="1"):
    # store predicted acoustic tokens
    acoustic_tokens = output.outputs[0].custom_outputs["acoustic_tokens"]  # T x 31
    step_acoustic_tokens = acoustic_tokens[-1:]  # 1 x 31
    acoustic_tokens_lst.append(step_acoustic_tokens)

    # if previously prepared input was last, break
    if i == next_subword_ids.shape[0] - 1:
        break

    current_subword_id = next_subword_ids[i:(i+1)]  # (1,)
    new_custom_inputs = {
        "input_acoustic_embeds": torch.randn(1, 4480, dtype=torch_type),
        "acoustic_tokens": step_acoustic_tokens,
        "context_text_tokens": context_subword_id,
        "text_mask": torch.ones_like(current_subword_id, dtype=torch_type),
        "bos_mask": torch.zeros_like(current_subword_id, dtype=torch_type),
    }
    context_subword_id = current_subword_id
    await engine.append_request(request_id="1", custom_inputs=new_custom_inputs)
    i += 1

acoustic_tokens_arr = torch.cat(acoustic_tokens_lst, dim=0)
torch.save(acoustic_tokens_arr.detach().cpu(), "pred_tokens.pt")